# BoreholeAI Quick Start

This notebook walks through using the BoreholeAI Python SDK to digitise borehole log PDFs.

You'll need an API key from [boreholeai.com/app/settings/api-keys](https://boreholeai.com/app/settings/api-keys).

## 1. Install

In [ ]:
!pip install boreholeai

## 2. Set up the client

In [ ]:
from boreholeai import BoreholeAI

# Get your API key at: https://boreholeai.com/app/settings/api-keys
client = BoreholeAI(api_key="bhai_your_api_key_here")

## 3. Process a single borehole log

In [ ]:
result = client.process_documents("BH01.pdf", output_dir="./results")

print(f"Status: {result.status}")
print(f"Job ID: {result.job_id}")
print(f"Pages processed: {result.num_pages}")
print(f"Credits used: {result.credits_used}")
print("\nOutput files:")
for f in result.files:
    print(f"  {f.filename} -> {f.path}")

## 4. Process a folder — fan-out + merge

Pass a directory and the SDK fans out one server-side job per file (up to `concurrency` in flight at a time, default 6). When all jobs finish, results are merged client-side into:

- `Borehole_ground_profile_merged.xlsx`
- `Borehole_test_data_merged.xlsx`
- `Borehole_ags4_merged.ags`
- one `*_annotated.pdf` per input file (always per-file)

Tune `concurrency` to your worker pool: smaller deployments → 3–5; larger → 10+.

In [ ]:
result = client.process_documents(
    "./borehole_logs/",
    output_dir="./results",
    concurrency=6,
)

print(f"Status: {result.status}")              # 'completed' | 'partial' | 'failed'
print(f"Successes: {result.successes}")
print(f"Failures:  {result.failures}")
print(f"Total pages: {result.num_pages}")
print(f"All job IDs: {result.job_ids}")
print("\nMerged outputs:")
for f in result.files:
    print(f"  {f.filename} -> {f.path}")

## 5. Work with the results

Output files are saved to `output_dir`. Single-file runs keep their original names (e.g. `Borehole_ground_profile.xlsx`); multi-file runs get a `_merged` suffix.

In [ ]:
import pandas as pd

# Multi-file run
ground_profile = pd.read_excel("./results/Borehole_ground_profile_merged.xlsx")
ground_profile.head()

In [ ]:
test_data = pd.read_excel("./results/Borehole_test_data_merged.xlsx")
test_data.head()

## 6. Resume after interrupt

If processing is interrupted (Ctrl-C, network drop, laptop sleep), simply re-run the same call with the same `output_dir`. The SDK persists per-file state to `.boreholeai_manifest.json` and skips work that's already done.

```python
# First run — interrupted halfway through 100 files
result = client.process_documents("./borehole_logs/", output_dir="./results")

# Same command, same output_dir — picks up where it left off
result = client.process_documents("./borehole_logs/", output_dir="./results")
```

## 7. Handle partial failures

If some files fail server-side processing (bad scan, unsupported format, etc.), the rest are merged normally and the failed files are reported via `result.failures`. The call only raises if every file fails.

In [ ]:
result = client.process_documents("./borehole_logs/", output_dir="./results")

if result.status == "partial":
    print(f"{len(result.failures)} file(s) failed:")
    for filename, error in result.failures.items():
        print(f"  {filename}: {error}")
    print(f"\n{len(result.successes)} file(s) succeeded and were merged.")

## 8. Error handling

In [ ]:
from boreholeai import AuthenticationError, InsufficientCreditsError

try:
    result = client.process_documents("borehole.pdf")
except AuthenticationError:
    print("Invalid API key")
except InsufficientCreditsError:
    print("Not enough credits - request more at support@boreholeai.com")